In [1]:
pip install pypdf numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 1 - install
!pip install PyPDF2 tiktoken google-generativeai

# Cell 2 - set key (never screenshot this cell)
import os
os.environ["GEMINI_API_KEY"] = "your_api_key_here"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.6/984.6 kB 9.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tiktoken]


In [8]:
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 11.2 MB/s  0:00:00


In [18]:
import os
import math
import tiktoken
import PyPDF2
from google import genai
from google.genai import types

file_path= "/Users/azeemkhalipha/ai-engineer-portfolio/Attention_Is_All_You_Need.pdf"

def load_pdf(file_path:str) ->str:
    read_pdf = PyPDF2.PdfReader(file_path)
    text = ""
    for page in read_pdf.pages:
        text += page.extract_text() or "" 
    return text




In [19]:
def chunk_text(text: str, window_size: int = 512, overlap: int = 50) -> list:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)  # converts string → list of token IDs
    chunks= []
    for i in range(0, len(tokens), window_size - overlap):
        chunk = tokens[i:i + window_size]
        chunks.append(tokenizer.decode(chunk))  # converts list of token IDs → string
    return chunks

In [29]:
def embed_chunks(chunks: list, client) -> list:
    store = []
    batch_size = 100
    for i in range(0, len(chunks), batch_size):
        batch= chunks[i:i+batch_size]
        response= client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=batch,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
        for j, embedding in enumerate(response.embeddings): 
            store.append({"chunk_id":i+j,
                          "text": batch[j],
                          "embedding": embedding.values})
    return store

In [22]:
def cosine_similarity(vec_a: list, vec_b: list) -> float:
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a ** 2 for a in vec_a))
    magnitude_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)

In [30]:
def search_chunks(query: str, store: list, client, top_k: int = 3) -> list:
    query_response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )
    query_embedding = query_response.embeddings[0].values

    results = []
    for item in store:
        similarity = cosine_similarity(query_embedding, item["embedding"])
        results.append({"chunk_id": item["chunk_id"], "text": item["text"], "similarity": similarity})

    # Sort results by similarity in descending order and return top_k
    results.sort(key=lambda x: x["similarity"], reverse=True)
    return results[:top_k]

In [32]:
if __name__ == "__main__":
    client= genai.Client()
    text= load_pdf(file_path)
    chunks= chunk_text(text)
    store= embed_chunks(chunks, client)
    query= "What is the transformer architecture?"
    results= search_chunks(query, store, client)
    for result in results:
        print(f"Chunk ID: {result['chunk_id']}, Similarity: {result['similarity']:.4f}")
        print(f"Text: {result['text']}\n")
    

Chunk ID: 3, Similarity: 0.7565
Text:  18] and [9].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [ 5,2,35].
Here, the encoder maps an input sequence of symbol representations (x1, ..., x n)to a sequence
of continuous representations z= (z1, ..., z n). Given z, the decoder then generates an output
sequence (y1, ..., y m)of symbols one element at a time. At each step the model is auto-regressive
[10], consuming the previously generated symbols as additional input when generating the next.
2Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N= 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism